# GTFN — orijinal protokol, test sızıntısı olmadan

**Ne değişti?** Eğitim kümesi, model ve eğitim orijinal GTFN ile birebir aynı. Tek fark, raporlanan epoch'un **test kümesine göre değil, ayrı bir doğrulama kümesine göre** seçilmesi.
Doğrulama pikselleri eğitim dışındaki piksellerden alınıyor (sınıf başına en fazla 10) ve test kümesinden çıkarılıyor.

Her koşu için üç sonuç kaydediliyor:
- `selected`: doğrulamaya göre seçilen epoch'un test sonucu → **raporlanacak sonuç**
- `last_epoch`: son epoch'un test sonucu
- `test_oracle_leaky`: orijinal yöntem (teste göre seçim) → sadece sızıntının büyüklüğünü ölçmek için

Sonuçlar Drive'a (`hsi-gnn-PHD/results/GTFN_noleak/`) ve W&B'ye yazılır.

**Sıra:** Hücreleri yukarıdan aşağıya çalıştır. Kesinti olursa 1–5 arası hücreleri yeniden çalıştırıp 7. hücreyle devam et; tamamlanmış koşular atlanır.

Her çalıştırma, Drive'daki sonuç klasöründe `DENEY_KAYDI.md` dosyasına otomatik bir giriş ekler. Deneyin amacını ve yorumunu repodaki `DENEYLER.md` dosyasına yaz.

## 1. GPU kontrolü

In [ ]:
!nvidia-smi -L

## 2. Drive'ı bağla

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

## 3. Kodu GitHub'dan çek
`GITHUB_USER` satırına kendi kullanıcı adını yaz.

In [ ]:
GITHUB_USER = "KULLANICI_ADIN"   # <-- değiştir

import os
os.chdir("/content")
if os.path.exists("/content/GTFN/.git"):
    %cd /content/GTFN
    !git pull -q
else:
    !git clone -q https://github.com/{GITHUB_USER}/GTFN.git /content/GTFN
    %cd /content/GTFN
!git log --oneline -3
!ls scripts run_utils.py

## 4. Veriyi Drive'dan kopyala

In [ ]:
import os, shutil
SRC = "/content/drive/MyDrive/hsi-gnn-PHD/HSI_data"
os.makedirs("data", exist_ok=True)
for f in ["Indian_pines_corrected.mat", "Indian_pines_gt.mat", "PaviaU.mat",
          "PaviaU_gt.mat", "Salinas_corrected.mat", "Salinas_gt.mat"]:
    shutil.copy(os.path.join(SRC, f), "data")
    print("✓", f, round(os.path.getsize(os.path.join("data", f)) / 2**20, 1), "MB")

## 5. Paketler ve W&B girişi
Anahtarı Colab Secrets'a `WANDB_API_KEY` adıyla kaydettiysen oradan okunur. Okunamazsa `wandb.login()` anahtarı sorar: wandb.ai/authorize sayfasından kopyalayıp yapıştır.

In [ ]:
!pip install -q einops wandb
import os, wandb
try:
    from google.colab import userdata
    os.environ["WANDB_API_KEY"] = userdata.get("WANDB_API_KEY")
    print("Anahtar Colab Secrets'tan alındı.")
except Exception as e:
    print("Secrets okunamadı, anahtar elle istenecek:", type(e).__name__)
wandb.login()

## 6. Kısa deneme (Indian, seed 0, 20 epoch) — yaklaşık 1 dakika
Ayrı bir klasöre yazar, asıl sonuçlara karışmaz.

In [ ]:
!python -u scripts/run_seeds.py \
    --out_root /content/drive/MyDrive/hsi-gnn-PHD/results/GTFN_noleak_deneme \
    --datasets Indian --seeds 0 --epoches 20 --wandb_project hsi-gtfn-deneme \
    --exp_id E002-deneme --note "Kurulum testi: 20 epoch, tek seed"

## 7. Tam deney: 3 veri seti × 10 seed × 200 epoch
Tahmini süre T4'te yaklaşık 3 saat (Indian ~30 dk, PaviaU ~1 sa, Salinas ~1,5 sa).
Bu sürede bilgisayar uyku moduna geçmemeli. Bağlantı koparsa 1–5'i yeniden çalıştırıp bu hücreyi tekrar başlat: bitmiş seed'ler atlanır.

In [ ]:
!python -u scripts/run_seeds.py \
    --out_root /content/drive/MyDrive/hsi-gnn-PHD/results/GTFN_noleak \
    --datasets Indian PaviaU Salinas --seeds 0-9 --wandb_project hsi-gtfn \
    --exp_id E002 --note "Orijinal protokol, test sızıntısız seçim"

## 8. Eski (sızıntılı) sonuçlarla karşılaştırma
Aynı seed'de eğitim kümesi birebir aynı olduğu için seed seed karşılaştırma yapılabilir.

In [ ]:
import json, glob, os
import numpy as np
OLD = "/content/drive/MyDrive/hsi-gnn-PHD/GTFN/output"
NEW = "/content/drive/MyDrive/hsi-gnn-PHD/results/GTFN_noleak"

for ds in ["Indian", "PaviaU", "Salinas"]:
    rows = []
    for seed in range(10):
        old_f = os.path.join(OLD, f"{ds}_seed{seed:02d}.json")
        new_f = sorted(glob.glob(os.path.join(NEW, ds, f"seed{seed:02d}_*", "metrics.json")))
        if not (os.path.exists(old_f) and new_f):
            continue
        old = json.load(open(old_f))
        new = json.load(open(new_f[-1]))
        rows.append((seed, old["OA"], new["test_oracle_leaky"]["OA"], new["selected"]["OA"], new["last_epoch"]["OA"]))
    if not rows:
        continue
    a = np.array(rows)
    print(f"\n{ds}  (n={len(a)})")
    print(" seed | eski OA | yeni-sızıntılı | yeni-seçili (val) | son epoch")
    for r in rows:
        print(f"  {r[0]:3d} |  {r[1]:6.2f} |     {r[2]:6.2f}     |      {r[3]:6.2f}       |  {r[4]:6.2f}")
    print(f" ort. |  {a[:,1].mean():6.2f} |     {a[:,2].mean():6.2f}     |      {a[:,3].mean():6.2f}       |  {a[:,4].mean():6.2f}")
    print(f" sızıntı farkı (eski − val seçili): {np.mean(a[:,1]-a[:,3]):+.2f} puan")